# Soil Grain-Size Distribution Prediction from Images

This notebook documents the image-based machine-learning workflow used in the project.
It extracts handcrafted image features, aggregates photographs by soil sample, evaluates
multi-output Ridge regression with Leave-One-Out Cross-Validation (LOOCV), predicts
cumulative grain-size curves for unseen sites, and creates a submission CSV.

**Dataset recorded in the original run:** 24 labeled soil samples, 127 training photographs,
35 test photographs, and 10 test sites.

> **Reproducibility note:** the original notebook is preserved at
> `notebooks/original_colab_notebook.ipynb`. This organized notebook consolidates the
> relevant workflow cells and adds explanatory markdown. You must provide the data files
> and image folders yourself; they are not bundled in this repository.

## 1. Environment and data paths

Run in Google Colab. The CSV files should be uploaded to `/content`, and the image folders
should be available in Google Drive. Update the two folder paths below if your Drive layout
differs from the original project.

In [ ]:
from pathlib import Path
import os
import re
import unicodedata
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content')
TRAIN_IMG_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Training-All_Photos_updated')
TEST_IMG_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Test_All_Photos')

train_path = BASE / 'Training_labels_updated.csv'
ppm_path = BASE / 'ppm_updated.csv'

for required_path in [train_path, ppm_path, TRAIN_IMG_DIR, TEST_IMG_DIR]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required project input not found: {required_path}')

train = pd.read_csv(train_path)
ppm = pd.read_csv(ppm_path)

print('Training labels:', train.shape)
print('Camera / PPM metadata:', ppm.shape)
print('Training image folder:', TRAIN_IMG_DIR)
print('Test image folder:', TEST_IMG_DIR)
display(train.head())
display(ppm.head())

## 2. Match training photographs to labeled samples

Training filenames contain sample identifiers such as `F827` and `G190`. The following
cell extracts those identifiers and checks that photographs and labels can be matched.

In [ ]:
from pathlib import Path
import re
import pandas as pd

TRAIN_IMG_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Training-All_Photos_updated"
)

# Extract sample IDs such as F827 and G190 from filenames
image_records = []

for path in TRAIN_IMG_DIR.iterdir():
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        match = re.search(r'(?<![A-Za-z0-9])([A-Z]\d{3})(?!\d)', path.name)

        if match:
            image_records.append({
                "sample_id": match.group(1),
                "filename": path.name
            })

image_df = pd.DataFrame(image_records)

# Count photographs for each sample
photo_counts = (
    image_df.groupby("sample_id")
    .size()
    .reset_index(name="number_of_photos")
    .sort_values("sample_id")
)

# Compare photo IDs with the labeled samples
label_ids = set(train["sample_id"].astype(str))
photo_ids = set(image_df["sample_id"])

print("Labeled samples:", len(label_ids))
print("Samples with matching photographs:", len(label_ids & photo_ids))
print("Photo IDs without labels:", sorted(photo_ids - label_ids))
print("Labels without photographs:", sorted(label_ids - photo_ids))

print("\nPhotographs per sample:")
display(photo_counts)

print("\nNumber of photos per sample:")
display(photo_counts["number_of_photos"].value_counts().sort_index())

## 3. Handcrafted image feature extraction

Each image is resized proportionally to fit within 512 × 512 pixels. The feature set includes
RGB and HSV channel means and standard deviations, grayscale percentiles and standard
deviation, Canny edge density, Laplacian variance, and normalized RGB histograms.

The original workflow extracts 44 numerical features per photograph and averages the
photographs belonging to the same labeled soil sample.

In [ ]:
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict

# Reuse TRAIN_IMG_DIR and image_df from the previous cell

def extract_features(image_path):
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        image.thumbnail((512, 512))
        rgb = np.asarray(image, dtype=np.uint8)

    # Convert RGB to other color representations
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    features = {}

    # Average color and color variation
    for name, array in [("rgb", rgb), ("hsv", hsv)]:
        for i in range(3):
            channel = array[:, :, i].astype(np.float32)
            features[f"{name}_mean_{i}"] = float(channel.mean())
            features[f"{name}_std_{i}"] = float(channel.std())

    # Brightness distribution
    for percentile in [10, 25, 50, 75, 90]:
        features[f"gray_p{percentile}"] = float(
            np.percentile(gray, percentile)
        )

    features["gray_std"] = float(gray.std())

    # Texture: edge density and local intensity changes
    edges = cv2.Canny(gray, 50, 150)
    features["edge_density"] = float((edges > 0).mean())

    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    features["laplacian_variance"] = float(laplacian.var())

    # Coarse color histograms
    for channel in range(3):
        hist = cv2.calcHist(
            [rgb], [channel], None, [8], [0, 256]
        ).flatten()
        hist = hist / max(hist.sum(), 1)

        for bin_idx, value in enumerate(hist):
            features[f"rgb_hist_{channel}_{bin_idx}"] = float(value)

    return features


# Extract features for every training photograph
records = []

for _, row in image_df.iterrows():
    path = TRAIN_IMG_DIR / row["filename"]

    try:
        features = extract_features(path)
        features["sample_id"] = row["sample_id"]
        records.append(features)
    except Exception as error:
        print("Could not process:", path.name, error)

photo_features = pd.DataFrame(records)

print("Photographs processed:", len(photo_features))
print("Features per photograph:", photo_features.shape[1] - 1)

# Combine multiple photographs belonging to the same sample.
# The mean gives each sample one feature vector.
sample_features = (
    photo_features.groupby("sample_id")
    .mean(numeric_only=True)
    .sort_index()
)

# Align labels to exactly the same sample order
labels = train.set_index("sample_id")
sample_features = sample_features.loc[
    sample_features.index.intersection(labels.index)
]
labels = labels.loc[sample_features.index]

print("Sample-level feature matrix:", sample_features.shape)
print("Label matrix:", labels.shape)

display(sample_features.head())

## 4. Baseline and Ridge regression evaluation

The target consists of 11 cumulative percentages at grain-size support points from
0.002 mm to 200 mm. LOOCV holds out one soil sample at a time.

`log_weighted_curve_error` is a project-specific proxy: normalized area between cumulative
curves on the logarithmic grain-size axis. It is not asserted to be the competition's exact
scoring implementation.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict

# Grain-size support points, in millimeters
sizes = np.array([
    0.002, 0.0063, 0.02, 0.063, 0.2, 0.63,
    2, 6.3, 20, 63, 200
])

X = sample_features.to_numpy(dtype=float)
# Match the exact column names in your training CSV
target_columns = [
    "0.002", "0.0063", "0.02", "0.063", "0.2", "0.63",
    "2", "6.3", "20", "63", "200"
]

X = sample_features.to_numpy(dtype=float)
Y = labels.loc[sample_features.index, target_columns].to_numpy(dtype=float)

print("Feature matrix:", X.shape)
print("Target matrix:", Y.shape)

# Use log grain size to define the interval weights.
# This is a useful proxy metric; verify it against the competition's
# exact scoring implementation if one is available.
log_sizes = np.log(sizes)

def make_valid(pred):
    """Keep predictions between 0 and 100, monotonic, and ending at 100."""
    pred = np.clip(pred, 0, 100)
    pred = np.maximum.accumulate(pred, axis=1)
    pred[:, -1] = 100.0
    return pred

def log_weighted_curve_error(y_true, y_pred):
    """Normalized area between cumulative curves on log-size axis."""
    y_pred = make_valid(np.asarray(y_pred).copy())
    errors = np.abs(np.asarray(y_true) - y_pred)
    return np.trapz(errors, x=log_sizes, axis=1).mean() / (
        log_sizes[-1] - log_sizes[0]
    )

cv = LeaveOneOut()

models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Ridge regression": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )
}

results = []
oof_predictions = {}

for name, model in models.items():
    # Every sample is predicted without using its own label in training.
    predictions = cross_val_predict(model, X, Y, cv=cv)
    predictions = make_valid(predictions)
    oof_predictions[name] = predictions

    results.append({
        "model": name,
        "log_weighted_curve_error": log_weighted_curve_error(Y, predictions),
        "mean_absolute_error": np.mean(np.abs(Y - predictions))
    })

results_df = pd.DataFrame(results).sort_values("log_weighted_curve_error")
display(results_df)

# Inspect actual vs predicted values for one held-out sample
sample_position = 0
comparison = pd.DataFrame({
    "grain_size_mm": sizes,
    "actual": Y[sample_position],
    "mean_baseline": oof_predictions["Mean baseline"][sample_position],
    "ridge": oof_predictions["Ridge regression"][sample_position]
})

display(comparison.round(2))

## 5. Ridge regularization search

The original experiment compared alpha values 0.1, 1, 10, and 100 using LOOCV. The
recorded best value was alpha = 10.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneOut, cross_val_predict

cv = LeaveOneOut()
alphas = [0.1, 1, 10, 100]

def valid_predictions(pred):
    pred = np.clip(pred.copy(), 0, 100)
    pred = np.maximum.accumulate(pred, axis=1)
    pred[:, -1] = 100.0
    return pred

def curve_error(y_true, y_pred):
    pred = valid_predictions(np.asarray(y_pred))
    errors = np.abs(np.asarray(y_true) - pred)
    return np.trapezoid(errors, x=np.log(sizes), axis=1).mean() / (
        np.log(sizes[-1]) - np.log(sizes[0])
    )

alpha_results = []
predictions_by_alpha = {}

for alpha in alphas:
    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha)
    )

    pred = cross_val_predict(model, X, Y, cv=cv)
    pred = valid_predictions(pred)
    predictions_by_alpha[alpha] = pred

    alpha_results.append({
        "alpha": alpha,
        "curve_error": curve_error(Y, pred),
        "mean_absolute_error": np.mean(np.abs(Y - pred))
    })

alpha_results = pd.DataFrame(alpha_results).sort_values("curve_error")
display(alpha_results.round(3))

# Show how the alpha=10 model performs on each held-out sample
pred = predictions_by_alpha[10]

per_sample = pd.DataFrame({
    "sample_id": sample_features.index,
    "MAE": np.mean(np.abs(Y - pred), axis=1),
    "curve_error": np.trapezoid(
        np.abs(Y - pred), x=np.log(sizes), axis=1
    ) / (np.log(sizes[-1]) - np.log(sizes[0]))
}).sort_values("MAE", ascending=False)

display(per_sample.round(2))

## 6. Camera metadata and training/test camera differences

Camera names are inferred from filename prefixes. PPM metadata is displayed for context.
Camera differences matter because image color and texture features can change across
devices. In the recorded data, training images primarily came from Motorola and Samsung
phones, while test images came from iPhone 14 / iPhone 16 devices.

In [ ]:
import re
import pandas as pd

# Extract the camera/model name from each filename
def get_camera_name(filename):
    match = re.search(
        r'(?<![A-Za-z0-9])([A-Z]\d{3})(?!\d)',
        filename
    )
    if match:
        return filename[:match.start()].rstrip("_- ").replace("_", " ")
    return "UNKNOWN"

camera_df = image_df.copy()
camera_df["camera_name"] = camera_df["filename"].apply(get_camera_name)

print("Training photos by camera:")
display(
    camera_df.groupby("camera_name")
    .agg(
        number_of_photos=("filename", "count"),
        number_of_samples=("sample_id", "nunique")
    )
    .sort_values("number_of_photos", ascending=False)
)

print("\nPPM metadata:")
display(ppm)

print("\nExample filename-to-camera mapping:")
display(camera_df[["filename", "camera_name"]].head(12))

In [ ]:

# Count how many photos of each camera were taken for each soil sample
camera_by_sample = pd.crosstab(
    camera_df["sample_id"],
    camera_df["camera_name"]
)

print("Camera usage for each soil sample:")
display(camera_by_sample)

print("\nNumber of cameras used per sample:")
display(
    (camera_by_sample > 0).sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("number_of_cameras")
    .to_frame("number_of_samples")
)

print("\nCamera counts in the test images:")
import os

test_files = [
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

test_camera_counts = pd.Series([
    " ".join(f.split("_HPC_")[0].split("_"))
    if "_HPC_" in f else "UNKNOWN"
    for f in test_files
]).value_counts()

display(test_camera_counts.to_frame("number_of_test_photos"))


### Match each photo feature row to its filename

The original notebook first made a many-to-many join by sample ID. The corrected version
below pairs photos using a per-sample photo counter and validates a one-to-one join.

In [ ]:

# Match each photo feature row to one filename
features_match = photo_features.copy()
files_match = image_df[["sample_id", "filename"]].copy()

features_match["photo_number"] = (
    features_match.groupby("sample_id").cumcount()
)
files_match["photo_number"] = (
    files_match.groupby("sample_id").cumcount()
)

photo_camera_df = features_match.merge(
    files_match,
    on=["sample_id", "photo_number"],
    how="left",
    validate="one_to_one"
)

print("Photo feature rows:", len(photo_features))
print("Matched rows:", len(photo_camera_df))
print("Missing filenames:", photo_camera_df["filename"].isna().sum())

photo_camera_df["camera_name"] = photo_camera_df["filename"].apply(
    get_camera_name
)

print("\nPhotos matched by camera:")
display(photo_camera_df["camera_name"].value_counts().to_frame("photos"))

print("\nExample matched rows:")
display(
    photo_camera_df[
        ["sample_id", "photo_number", "filename", "camera_name"]
    ].head(15)
)


In [ ]:

candidate_features = [
    "rgb_mean_0",
    "rgb_mean_1",
    "rgb_mean_2",
    "hsv_mean_0",
    "hsv_mean_1",
    "hsv_mean_2",
    "gray_p50",
    "gray_std",
    "edge_density",
    "laplacian_variance",
]

print("Average features by camera (127 matched photos):")
display(
    photo_camera_df.groupby("camera_name")[candidate_features]
    .mean()
    .round(2)
    .T
)

print("\nNumber of distinct soil samples represented per camera:")
display(
    photo_camera_df.groupby("camera_name")["sample_id"]
    .nunique()
    .to_frame("number_of_samples")
)


## 7. Extract test-image features and aggregate by site

Test photographs are grouped by site name after removing camera prefixes and numbered
photo suffixes. The mean feature vector for each site is used for final prediction.

In [ ]:

import os
import pandas as pd

test_files = sorted([
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

test_df = pd.DataFrame({"filename": test_files})

def get_test_camera(filename):
    prefix = filename.split("_HPC_")[0] if "_HPC_" in filename else "UNKNOWN"
    return prefix.replace("_", " ")

test_df["camera_name"] = test_df["filename"].apply(get_test_camera)

print("Total test photos:", len(test_df))
print("\nTest photos by camera:")
display(test_df["camera_name"].value_counts().to_frame("number_of_photos"))

print("\nAll test filenames:")
display(test_df)


In [ ]:

import os
import pandas as pd

test_files = sorted([
    f for f in os.listdir(TEST_IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

test_photo_features = []

for filename in test_files:
    image_path = os.path.join(TEST_IMG_DIR, filename)
    features = extract_features(image_path)

    if features is not None:
        test_photo_features.append({
            "filename": filename,
            **features
        })

test_photo_features = pd.DataFrame(test_photo_features)

print("Test photos found:", len(test_files))
print("Test photos processed:", len(test_photo_features))
print("Feature table shape:", test_photo_features.shape)
print("\nFirst five rows:")
display(test_photo_features.head())


In [ ]:

import os
import re
import pandas as pd

def get_site_name(filename):
    # Remove the camera prefix and keep the site name
    stem = os.path.splitext(filename)[0]

    if "_HPC_" in stem:
        site = stem.split("_HPC_", 1)[1]
    else:
        site = stem

    # Remove photo-number suffixes such as (1), (2), etc.
    site = re.sub(r"\s*\(\d+\)$", "", site).strip()

    return site

test_photo_features["site_name"] = (
    test_photo_features["filename"].apply(get_site_name)
)

print("Photos per site:")
display(
    test_photo_features.groupby("site_name")
    .agg(
        number_of_photos=("filename", "count")
    )
    .sort_index()
)

# Average the 44 image features for each site
feature_columns = [
    col for col in photo_features.columns
    if col != "sample_id"
]

test_site_features = (
    test_photo_features.groupby("site_name")[feature_columns]
    .mean()
)

print("\nNumber of distinct test sites:", len(test_site_features))
print("Test site feature matrix shape:", test_site_features.shape)


## 8. Fit the final model and predict test sites

The selected model is `StandardScaler` followed by `Ridge(alpha=10.0)`, fitted on all
24 labeled samples. Predictions are constrained to be cumulative, between 0 and 100,
and exactly 100 at the largest grain-size support point.

In [ ]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd

# Ensure test features match the training feature order
X_train_df = sample_features.copy()
X_test_df = test_site_features[feature_columns].copy()

print("Training features:", X_train_df.shape)
print("Test features:", X_test_df.shape)
print("Same feature columns:", list(X_train_df.columns) == list(X_test_df.columns))

# Train the current best model on all 24 labeled samples
final_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10.0)
)

final_model.fit(X_train_df.to_numpy(dtype=float), Y)

# Predict the 11 cumulative grain-size values for each test site
raw_test_predictions = final_model.predict(
    X_test_df.to_numpy(dtype=float)
)

test_predictions = valid_predictions(raw_test_predictions)

prediction_df = pd.DataFrame(
    test_predictions,
    index=test_site_features.index,
    columns=target_columns
)

print("\nPrediction table shape:", prediction_df.shape)
display(prediction_df.round(2))


## 9. Within-site prediction variability

This is a consistency diagnostic: it measures how much predictions vary among photographs
from the same test site. Low variation does **not** establish prediction accuracy.

In [ ]:

# Predict each test photo individually
X_test_photos = test_photo_features[feature_columns].to_numpy(dtype=float)

photo_raw_predictions = final_model.predict(X_test_photos)
photo_predictions = valid_predictions(photo_raw_predictions)

photo_prediction_df = pd.DataFrame(
    photo_predictions,
    columns=target_columns
)

photo_prediction_df["site_name"] = test_photo_features["site_name"].to_numpy()

# Measure variation among photos from the same site
site_variability = photo_prediction_df.groupby("site_name")[target_columns].std()

variability_summary = pd.DataFrame({
    "number_of_photos": photo_prediction_df.groupby("site_name").size(),
    "average_prediction_std": site_variability.mean(axis=1),
    "maximum_prediction_std": site_variability.max(axis=1)
}).sort_values("average_prediction_std", ascending=False)

print("Prediction variation across photos of the same site:")
display(variability_summary.round(2))

print("\nOverall average within-site prediction standard deviation:",
      round(site_variability.mean().mean(), 2))


## 10. Compare feature averaging with prediction averaging

This LOOCV experiment compares (1) averaging image features before fitting/predicting and
(2) predicting each held-out photograph and averaging its predictions. The recorded results
favored feature averaging.

The `train_photo_df` assignment below makes the source notebook's dependency explicit.

In [ ]:
train_photo_df = photo_camera_df.copy()



from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import numpy as np

sample_ids = sample_features.index.to_list()
Y_all = labels.loc[sample_ids, target_columns].to_numpy(dtype=float)

loo = LeaveOneOut()
pred_feature_avg = np.zeros_like(Y_all)
pred_prediction_avg = np.zeros_like(Y_all)

for train_idx, val_idx in loo.split(sample_ids):
    train_ids = [sample_ids[i] for i in train_idx]
    val_id = sample_ids[val_idx[0]]
    val_position = val_idx[0]

    model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )
    model.fit(
        sample_features.loc[train_ids, feature_columns].to_numpy(dtype=float),
        labels.loc[train_ids, target_columns].to_numpy(dtype=float)
    )

    # Method 1: average features first
    x_avg = sample_features.loc[[val_id], feature_columns].to_numpy(dtype=float)
    pred_feature_avg[val_position] = valid_predictions(
        model.predict(x_avg)
    )[0]

    # Method 2: predict individual photos, then average predictions
    heldout_photos = train_photo_df[train_photo_df["sample_id"] == val_id]
    x_photos = heldout_photos[feature_columns].to_numpy(dtype=float)
    individual_preds = valid_predictions(model.predict(x_photos))

    averaged_curve = individual_preds.mean(axis=0)
    pred_prediction_avg[val_position] = valid_predictions(
        averaged_curve.reshape(1, -1)
    )[0]

print("Feature averaging:")
print("  Curve error:", round(curve_error(Y_all, pred_feature_avg), 3))
print("  MAE:", round(np.mean(np.abs(Y_all - pred_feature_avg)), 3))

print("\nPrediction averaging:")
print("  Curve error:", round(curve_error(Y_all, pred_prediction_avg), 3))
print("  MAE:", round(np.mean(np.abs(Y_all - pred_prediction_avg)), 3))


## 11. Build the competition submission

The site-name mapping converts the test-site names parsed from filenames to the exact
`sample_id` values used by the supplied sample-submission template. Name normalization
handles differences in accents, punctuation, spaces, and underscores.

In [ ]:

import pandas as pd
import numpy as np
import unicodedata
import re

site_mapping = {
    "HPC_Airbus BS6-3": "Airbus BS6-3",
    "HPC_Audorfring": "Audorfring",
    "HPC_Muenster_BS6_9_0-10m": "Münster_BS6_9,0-10m",
    "HPC_Airbus BS10-4bis7": "Airbus BS10-4bis7",
    "HPC_Airbus BS12-5bis8": "Airbus BS12-5bis8",
    "HPC_Kleinkummerfeld 2-2": "Kleinkummerfeld 2-2",
    "HPC_Kleinkummerfeld 2-3": "Kleinkummerfeld 2-3",
    "HPC_Kleinkummerfeld 9-4": "Kleinkummerfeld 9-4",
    "HPC_Kleinkummerfeld 18-3": "Kleinkummerfeld 18-3",
    "HPC_Testfeld Lidl WHV": "Testfeld Lidl WHV"
}

def normalize_name(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(c for c in value if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", value.lower())

pred_lookup = prediction_df.copy()

if "site_name" in pred_lookup.columns:
    pred_lookup = pred_lookup.set_index("site_name")

# Match normalized names, so accents, spaces, commas and underscores
# do not prevent a match.
normalized_index = {}
for name in pred_lookup.index:
    key = normalize_name(name)
    if key in normalized_index:
        raise ValueError(f"Ambiguous normalized site name: {name}")
    normalized_index[key] = name

rows = []

for sample_id, expected_name in site_mapping.items():
    key = normalize_name(expected_name)

    if key not in normalized_index:
        print("Available names:", pred_lookup.index.tolist())
        raise ValueError(f"Could not match site: {expected_name}")

    actual_name = normalized_index[key]
    values = pred_lookup.loc[actual_name, target_columns].to_numpy(dtype=float)

    row = {"sample_id": sample_id}
    row.update(dict(zip(target_columns, values)))
    rows.append(row)

submission = pd.DataFrame(rows, columns=["sample_id"] + target_columns)

assert submission.shape == (10, 12)
assert submission["sample_id"].is_unique
assert np.isfinite(submission[target_columns].to_numpy(dtype=float)).all()
assert (submission[target_columns].to_numpy(dtype=float) >= 0).all()
assert (submission[target_columns].to_numpy(dtype=float) <= 100).all()

submission.to_csv("/content/submission.csv", index=False)

print("Submission created successfully!")
print("Shape:", submission.shape)
display(submission.round(2))


## 12. Validate and download the CSV

The final file should have 10 rows and 12 columns, no missing values, unique sample IDs,
non-decreasing cumulative values, and a final `200` column equal to 100.

In [ ]:

# Reload the actual saved file and verify every row and value.
check = pd.read_csv("/content/submission.csv")

print("Shape:", check.shape)
print("Missing values:", check.isna().sum().sum())
print("Duplicate sample IDs:", check["sample_id"].duplicated().sum())
print("\nLast row:")
display(check.tail(1))

assert check.shape == (10, 12)
assert check.isna().sum().sum() == 0
assert check["sample_id"].duplicated().sum() == 0
assert np.all(np.diff(check[target_columns].to_numpy(dtype=float), axis=1) >= -1e-9)
assert np.allclose(check["200"].to_numpy(dtype=float), 100)

print("\nAll submission format checks passed.")


In [ ]:
from google.colab import files
files.download('/content/submission.csv')

## Recorded experiment results

| Method | Log-weighted curve-error proxy | MAE |
|---|---:|---:|
| Dummy mean baseline | 17.821 | 16.343 |
| Ridge, alpha = 10, feature averaging | **7.633** | **7.057** |
| Ridge, alpha = 10, prediction averaging | 8.145 | 7.523 |

These values are from the recorded notebook run and the follow-up experiment in the project
conversation. They are not a guarantee of test-set performance.

## Limitations

- Only 24 labeled soil samples are available, so generalization estimates are uncertain.
- LOOCV holds out samples, but does not fully simulate the training-to-test camera shift.
- The image features are handcrafted appearance statistics, not direct grain segmentation.
- The curve-error calculation is a proxy unless verified against the official competition metric.
- Predictions that quickly reach 100% or contain flat sections should be interpreted cautiously.